In [3]:
# --- import libraries ---
import pandas as pd
import numpy as np
import os
import sys
import shutil
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from sklearn.preprocessing import StandardScaler
from typing import Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import ElasticNet
from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxnetSurvivalAnalysis
import matplotlib.pyplot as plt

sys.path.append('../utilities')

from preprocess import preprocess

PARTICIPANT_DATA_PATH = '../participant_data/'

In [4]:
# Define path to the single participant data folder.
PARTICIPANT_DATA_PATH = '../participant_data/'
SUBMISSION_DIR = 'my_prediction_submission_elasticnet'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Define all 16 event pairs
index_events = ["Borrow", "Deposit", "Repay", "Withdraw"]
outcome_events = index_events + ["Liquidated"]
event_pairs = []
for index_event in index_events:
    for outcome_event in outcome_events:
        if index_event == outcome_event:
            continue
        event_pairs.append((index_event, outcome_event))

for index_event, outcome_event in event_pairs:
    print(f"\n{'='*50}")
    print(f"Processing and Predicting for: {index_event} -> {outcome_event}")
    print(f"{'='*50}")
    
    dataset_path = os.path.join(index_event, outcome_event)
    
    # --- Load and Preprocess ---
    try:
        train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
        # the test features for participants are the validation features.
        test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))
    except FileNotFoundError as e:
        print(f"Data not found for {dataset_path}. Skipping.")
        continue
        
    X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)

    # --- Train Model ---
    try:
        lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
        # prepare data
        X = lifelines_train_df.drop(['timeDiff', 'status'], axis=1)

        ## below line takes the status and timeDiff columns from the DataFrame, converts them into a structured NumPy array y where:
        # status is stored as a boolean (event occurred or not).
        # timeDiff is stored as a float (time until event or censoring).

        y = np.array([(bool(event), time) for event, time in 
                    zip(lifelines_train_df['status'], lifelines_train_df['timeDiff'])], 
                    dtype=[('status', '?'), ('timeDiff', '<f8')])

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # define CoxNet Elastic Net
        model = CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01)

        try:
            # First attempt to fit the model with standard parameters
            model.fit(X_train, y_train)
        except ConvergenceError:
            # If it fails, try again with a stronger penalizer and smaller step size
            print("  - Standard model failed to converge. Trying robust parameters...")
            model = CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01)
            model.fit(X_train, y_train)
            print("  - Successfully converged with robust parameters.")

        # --- Generate and Save Predictions ---
        print(f"Generating predictions for {dataset_path}...")

        # predictions (risk scores)
        risk_test = model.predict(X_test)
        c_index_test = concordance_index_censored(y_test['status'], y_test['timeDiff'], risk_test)[0]
        print(f"Test c-index: {c_index_test:.3f}")

        predictions = -model.predict(X_test_processed) # higher values mean lower survival probability (higher hazard)
        print(f"Predictions generated for X_test_processed")
        
        # Save predictions to a CSV file
        prediction_filename = dataset_path.replace(os.sep, '_') + '.csv'
        prediction_save_path = os.path.join(SUBMISSION_DIR, prediction_filename)
        pd.DataFrame(predictions).to_csv(prediction_save_path, header=False, index=False)
        print(f"  -> Predictions saved to {prediction_save_path}\n")
        
    except (ConvergenceError, ValueError) as e:
        print(f"\nERROR: The model for {dataset_path} failed to train even with robust parameters. No prediction file will be created.")
        print(f"Details: {e}")

print("\n\nAll prediction files have been generated.")


Processing and Predicting for: Borrow -> Deposit
Generating predictions for Borrow/Deposit...
Test c-index: 0.721
Predictions generated for X_test_processed
  -> Predictions saved to my_prediction_submission_elasticnet/Borrow_Deposit.csv


Processing and Predicting for: Borrow -> Repay
Generating predictions for Borrow/Repay...
Test c-index: 0.788
Predictions generated for X_test_processed
  -> Predictions saved to my_prediction_submission_elasticnet/Borrow_Repay.csv


Processing and Predicting for: Borrow -> Withdraw
Generating predictions for Borrow/Withdraw...
Test c-index: 0.715
Predictions generated for X_test_processed
  -> Predictions saved to my_prediction_submission_elasticnet/Borrow_Withdraw.csv


Processing and Predicting for: Borrow -> Liquidated
Generating predictions for Borrow/Liquidated...
Test c-index: 0.829
Predictions generated for X_test_processed
  -> Predictions saved to my_prediction_submission_elasticnet/Borrow_Liquidated.csv


Processing and Predicting for: De

## Step 4: Create the Submission ZIP file

The final step is to package all 16 of your prediction CSV files into a single `submission.zip` file.

In [6]:
output_zip_filename = 'submission_elasticnet'
shutil.make_archive(output_zip_filename, 'zip', SUBMISSION_DIR)

print(f"Successfully created '{output_zip_filename}.zip' from the '{SUBMISSION_DIR}' directory.")
print("You can now upload this file to the CodaBench competition.")

Successfully created 'submission_elasticnet.zip' from the 'my_prediction_submission_elasticnet' directory.
You can now upload this file to the CodaBench competition.
